## Deploying Flixtube to a Local Kubernetes Cluster

In this notebook, we look at how we can deploy the Flixtube microservice application to Docker Desktop's Kubernetes cluster.

- Make sure you have a Kubernetes cluster (Docker Desktop) running.
- Also make sure you have installed the `kubectl` tool on your computer.

---

## Architecture

The `Flixtube` application is a microservice-based application with seven custom microservices:
  - `Web`: A Blazor frontend service with a user interface.
  - `Gateway`: An API gateway service, which the `Web` service talks to, and which redirects requests to the various other services below.
  - `Metadata`: A metadata service that manages video metadata (e.g. the name of a video).
  - `History`: A history service that manages video viewing history (e.g. the name, date and time of when a video was played).
  - `VideoUpload`: A service used to upload a video from the `Web` frontend via the `Gateway` to the `VideoStorage` service.
  - `VideoStorage`: A service responsible for storing video files.
  - `Videostreaming`: A service used to stream a video from the `VideoStorage` service via the `Gateway` to the `Web` service.

The `Flixtube` application uses a number of **backing services**:
  - `Sql Server`: An SQL Server service to store relational data.
  - `Minio` or `Azure Storage`: A Minio or Azure Storage service to store Blob data (video files).
  - `Rabbit MQ`: A Rabbit Message Queue (MQ) service for publisher/subscriber-based messaging.

<img src="notebook_images/flixtube_architecture.drawio.png"  width="800" alt="Flixtube Architecture" />

**Communication**

- The **Browser** communicates with:
  - The `Web` microservice via HTTP.
  - The `Gateway` microservice via HTTP when streaming (playing) a Video.
- The `Gateway` communicates with the microservices below via HTTP:
  - `Web`, `Metadata`, `History`, `VideoStreaming`, `VideoUpload`, and `VideoStorage`.
- `VideoStorage` communicates with:
  - The `Gateway`, `VideoStreaming`, and `VideoUpload` microservices via HTTP.
  - A storage backing service which can be one of the below:
    - `Minio` via HTTP if the `VideoStorage` microservice is built using the `MinioStorage` VSCode project.
    - `Azure Storage` via HTTP if the `VideoStorage` microservice is built using the `AzureStorage` VSCode project.
- `VideoUpload` communicates with:
  - The `Gateway` and `VideoStorage` microservices via HTTP.
  - A `RabbitMQ` backing service via AMQP, where it publishes `VideoUploaded` messages to an Exchange called `uploaded`.
- `VideoStreaming` communicates with:
  - The `Gateway` and `VideoStorage` microservices via HTTP.
  - A `RabbitMQ` backing service via AMQP, where it publishes `VideoViewed` messages to an Exchange called `viewed`.
- `Metadata` communicates with:
  - The `Gateway` microservice via HTTP.
  - An `SQLServer` backing service via HTTP, where it manages video metadata in a `Videos` table. 
  - A `RabbitMQ` backing service via AMQP, where it subscribes to an Exchange called `uploaded` receiving `VideoUploaded` messages.
- `History` communicates with:
  - The `Gateway` microservice via HTTP.
  - An `SQLServer` backing service via HTTP, where it manages video viewing history in a `ViewHistorys` table. 
  - A `RabbitMQ` backing service via AMQP, where it subscribes to an Exchange called `viewed` receiving `VideoViewed` messages.

---

## Deploying Flixtube to Docker Desktop's Kubernetes Cluster

### Run the command below to build the Docker images for the Flixtube application

- The command used here is `docker build -t <ImageName>:<Tag> --file <PathToDockerFile> <PathToFolderWithFiles>`
- For example, for the `Flixtube.MinioStorage` microservice:
  - `<ImageName>` is `video-storage`
  - `<Tag>` is `1`
  - `<PathToDockerFile>` relative to this notebook is `../../flixtube/Flixtube.MinioStorage/Dockerfile-prod`
  - `<PathToFolderWithFiles>` relative to this notebook is `../../flixtube/Flixtube.MinioStorage`

Note that we could replace `Dockerfile-prod` with `Dockerfile-dev` if we wanted to build the development versions of the images. 

In [5]:
# !docker build -t video-storage:1 --file ../../flixtube/Flixtube.AzureStorage/Dockerfile-prod ../../flixtube/Flixtube.AzureStorage
!docker build -t video-storage:1 --file ../../flixtube/Flixtube.MinioStorage/Dockerfile-prod ../../flixtube/Flixtube.MinioStorage
!docker build -t video-upload:1 --file ../../flixtube/Flixtube.VideoUpload/Dockerfile-prod ../../flixtube/Flixtube.VideoUpload
!docker build -t video-streaming:1 --file ../../flixtube/Flixtube.VideoStreaming/Dockerfile-prod ../../flixtube/Flixtube.VideoStreaming
!docker build -t metadata:1 --file ../../flixtube/Flixtube.Metadata/Dockerfile-prod ../../flixtube/Flixtube.Metadata
!docker build -t history:1 --file ../../flixtube/Flixtube.History/Dockerfile-prod ../../flixtube/Flixtube.History
!docker build -t gateway:1 --file ../../flixtube/Flixtube.Gateway/Dockerfile-prod ../../flixtube/Flixtube.Gateway
!docker build -t web:1 --file ../../flixtube/Flixtube.Web/Dockerfile-prod ../../flixtube/Flixtube.Web

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile-prod
#1 transferring dockerfile: 462B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for mcr.microsoft.com/dotnet/aspnet:9.0
#2 DONE 0.4s

#3 [internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0
#3 DONE 0.4s

#4 [internal] load .dockerignore
#4 transferring context: 393B 0.0s done
#4 DONE 0.0s

#5 [build 1/6] FROM mcr.microsoft.com/dotnet/sdk:9.0@sha256:84fd557bebc64015e731aca1085b92c7619e49bdbe247e57392a43d92276f617
#5 DONE 0.0s

#6 [stage-1 1/3] FROM mcr.microsoft.com/dotnet/aspnet:9.0@sha256:07dd7f0c45263fee87e094b1e627b33a095f75c54be39c495de23b82b0936b9e
#6 DONE 0.0s

#7 [internal] load build context
#7 transferring context: 15.29kB 0.0s done
#7 DONE 0.0s

#8 [build 2/6] WORKDIR /src
#8 CACHED

#9 [build 3/6] COPY ./Flixtube.MinioStorage/*.csproj ./
#9 DONE 0.0s

#10 [build 4/6] RUN dotnet restore
#10 1.283   Determining projects to restore...
#10 6.014  

### List all Images on the Local Computer

- Notice that all the images have been built.

In [ ]:
!docker images

REPOSITORY                                           TAG                                                                           IMAGE ID       CREATED         SIZE
web                                                  1                                                                             2fee68db51b1   4 minutes ago   236MB
gateway                                              1                                                                             b3ca8fd84f68   5 minutes ago   227MB
history                                              1                                                                             f71aac9dc89a   6 minutes ago   299MB
video-streaming                                      1                                                                             687b8f684bbe   7 minutes ago   224MB
video-upload                                         1                                                                             323d5f63f86b   8 minutes ago  

### Run the command below to deploy the microservices to Docker Desktop's Kubernetes Cluster

- First we ensure `kubectl` is communicating with the `docker-desktop` Kubernetes cluster via the command:
  - kubectl config use-context docker-desktop
- Then, the main command used here is `kubectl apply -f <PathToYAMLManifestFile>`
  - For example, for the `Flixtube.MinioStorage` microservice's YAML manifest file containg its deployment and service definitions:
    - `<PathToYAMLManifestFile>` relative to this notebook is `../../flixtube/scripts/local-k8-cluster/minio.yaml`

In [18]:
!kubectl config use-context docker-desktop
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/rabbit.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/sqlserver.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/minio.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/video-storage.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/video-upload.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/video-streaming.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/metadata.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/history.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/gateway.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/web.yaml

Switched to context "docker-desktop".
deployment.apps/rabbit created
service/rabbit created
deployment.apps/db created
service/db created
deployment.apps/minio created
service/minio created
job.batch/minio-mc created
deployment.apps/video-storage created
service/video-storage created
deployment.apps/video-upload created
service/video-upload created
deployment.apps/video-streaming created
service/video-streaming created
deployment.apps/metadata created
service/metadata created
deployment.apps/history created
service/history created
deployment.apps/gateway created
service/gateway created
deployment.apps/web created
service/web created


### List all Pods, Deployments and Services running in Docker Desktop's Kubernetes Cluster

- We see that all the resources have been deployed to the Kubernetes cluster.
  - Notice that the `Web` microservice is available via a LoadBalancer on port `4000` (http://localhost:4000).
  - Notice that the `Gateway` microservice is available via a LoadBalancer on port `4010` (http://localhost:4010).

**Note that you might have to wait a while (30-45 second) before all the Pods are READY.**

In [23]:
!kubectl get pods,deployments,services -o wide

NAME                                  READY   STATUS    RESTARTS   AGE     IP          NODE             NOMINATED NODE   READINESS GATES
pod/db-6687ff9b86-cgh44               1/1     Running   0          3m1s    10.1.1.71   docker-desktop   <none>           <none>
pod/gateway-7c6cd8987c-mdxb4          1/1     Running   0          2m57s   10.1.1.79   docker-desktop   <none>           <none>
pod/history-649768dc8c-8hd27          1/1     Running   0          2m58s   10.1.1.78   docker-desktop   <none>           <none>
pod/metadata-74fbb48f7d-tr7w5         1/1     Running   0          2m59s   10.1.1.77   docker-desktop   <none>           <none>
pod/minio-768cf4767d-wz7zp            1/1     Running   0          3m1s    10.1.1.72   docker-desktop   <none>           <none>
pod/minio-mc-m228g                    1/1     Running   0          3m1s    10.1.1.73   docker-desktop   <none>           <none>
pod/rabbit-d57559b69-xgvqc            1/1     Running   0          3m1s    10.1.1.70   docker-d

---

## Using the Flixtube Application

- Visit the Home page http://localhost:4000

  - The Home page displays a list of videos (currently empty) with an `Id` and a `Name` column.
  - There are two buttons `Upload Video` and `Show Viewing History` at the top of the page.
    - The `Upload Video` button is used for uploading a video.
    - The `Show Viewing History` button is used to display a list of viewing history, i.e. when a video was viewed (played).
  - Click the `Upload Video` button on the Home page which will navigate to the Upload Video page.

<img src="notebook_images/flixtube_home_page.png"  width="600" alt="Flixtube Home Page" />

- The Upload Video page.

  - The Upload Video page allows the user to upload a video from the user's local file system.
  - There are four buttons `Home`, `Choose File`, `Upload` and `Cancel` on the page.
    - The `Home` button navigates back to the Home page.
    - The `Choose File` button pops up a window, allowing the user to choose a video file to upload.
      - The name of the chosen video file is displayed in the field to the right of the `Choose File` button.
    - The `Upload` button uploads the chosen video file.
    - The `Cancel` button returns the user to the Home page without uploading any video file.
  - Click the `Choose File` button and choose a video file.
    - There is a sample file in `workshop4/05_flixtube/SampleVideo_1280x720_1mb.mp4` you can use.
  - Then click the `Upload` button, which will upload the file and navigate back to the Home page.

<img src="notebook_images/flixtube_upload_page.png"  width="600" alt="Flixtube Upload Video Page" />

- Back on the Home page.
  - The Home page now displays one video in its a list of videos.
    - The `Id` column contains the video's GUID.
    - The `Name` column contains the video's name.
    - The blue `Play` button is used to play (view) the video.
    - The red `Delete` button is used to remove (delete) the video.
  - We could also access the Gateway to get the JSON document via the REST API.
    - http://localhost:4010/api/metadata
- Click the blue `Play` button which will navigate to the Play Video page.

<img src="notebook_images/flixtube_home_page2.png"  width="600" alt="Flixtube Home Page" />

- The Play Video page.

  - The Play Video page starts streaming the video to the browser.
    - The video's name is shown above the video.
    - Video controls for playing, pausing, etc. are shown below the video.
    - There is a `Home` button at the top of the page, which will navigate back to the Home page.
  - Click the `Home` button to navigate back to the Home page.

<img src="notebook_images/flixtube_play_page.png"  width="600" alt="Flixtube Play Video Page" />

- Back on the Home page.

  - Click the `Show Viewing History` button which will navigate to the Viewing History page.

<img src="notebook_images/flixtube_home_page2.png"  width="600" alt="Flixtube Home Page" />

- The Viewing History page.
  - The Viewing History page displays a list video viewings with an `Id` and a `ViewedAt` column.
    - The `Id` column contains the video's GUID.
    - The `ViewedAt` column contains the date and time the video was viewed (played).
    - There is a `Home` button at the top of the page, which will navigate back to the Home page.
  - We could also access the Gateway to get the JSON document via the REST API.
    - http://localhost:4010/api/history
  - Click the `Home` button to navigate back to the Home page.

<img src="notebook_images/flixtube_history_page.png"  width="600" alt="Flixtube Viewing History Page" />

---

## Removing Flixtube from Docker Desktop's Kubernetes Cluster

### Run the command below to delete the microservices from Docker Desktop's Kubernetes Cluster

- First we ensure `kubectl` is communicating with the `docker-desktop` Kubernetes cluster via the command:
  - kubectl config use-context docker-desktop
- Then, the main command used here is `kubectl delete -f <PathToYAMLManifestFile>`
  - For example, for the `Flixtube.MinioStorage` microservice's YAML manifest file containg its deployment and service definitions:
    - `<PathToYAMLManifestFile>` relative to this notebook is `../../flixtube/scripts/local-k8-cluster/minio.yaml`

In [24]:
!kubectl config use-context docker-desktop
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/rabbit.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/sqlserver.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/minio.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/video-storage.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/video-upload.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/video-streaming.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/metadata.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/history.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/gateway.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/web.yaml

Switched to context "docker-desktop".
deployment.apps "rabbit" deleted
service "rabbit" deleted
deployment.apps "db" deleted
service "db" deleted
deployment.apps "minio" deleted
service "minio" deleted
job.batch "minio-mc" deleted
deployment.apps "video-storage" deleted
service "video-storage" deleted
deployment.apps "video-upload" deleted
service "video-upload" deleted
deployment.apps "video-streaming" deleted
service "video-streaming" deleted
deployment.apps "metadata" deleted
service "metadata" deleted
deployment.apps "history" deleted
service "history" deleted
deployment.apps "gateway" deleted
service "gateway" deleted
deployment.apps "web" deleted
service "web" deleted


### List all Pods, Deployments and Services running in Docker Desktop's Kubernetes Cluster

- We see that all the resources have been removed from the Kubernetes cluster.
  - The only service displayed in the list should be the `service/kubernetes` service which we should never delete.

In [26]:
!kubectl get pods,deployments,services -o wide

NAME                 TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE   SELECTOR
service/kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   27d   <none>


### Run the command below to remove the Docker images for the Flixtube application

- The command used here is `docker rmi <ImageName>:<Tag> --force`
- For example, for the `Flixtube.MinioStorage` microservice:
  - `<ImageName>` is `video-storage`
  - `<Tag>` is `1`

In [27]:
!docker rmi video-storage:1 --force
!docker rmi video-upload:1 --force
!docker rmi video-streaming:1 --force
!docker rmi metadata:1 --force
!docker rmi history:1 --force
!docker rmi gateway:1 --force
!docker rmi web:1 --force

Untagged: video-storage:1
Deleted: sha256:057610f6e4bc4941c560718681ba19beb0c146f6dfa22f9c8877b5a2077b213d
Untagged: video-upload:1
Deleted: sha256:323d5f63f86b8e7a78705489b1e46b2bfb19d9ee1fbe171ab3a49b001e9cc111
Untagged: video-streaming:1
Deleted: sha256:687b8f684bbe9fd0198ac48a79cc2033f5b2d9c65208608d3588942c6cc5bf69
Untagged: metadata:1
Deleted: sha256:f3e63f316ba9a312032ad3221bf0c8535f0cb15e864af8d17d3106cbea3da0f3
Untagged: history:1
Deleted: sha256:f71aac9dc89ae0aba927b2cc3cb9d91054d74ee4ed8a946027435fd1cbe81e9d
Untagged: gateway:1
Deleted: sha256:b3ca8fd84f680d6f0fd3be33bd79c0a56ba40cc3963ab4cfe46113da204b2b48
Untagged: web:1
Deleted: sha256:2fee68db51b1cbf280cbab668528d564c192e9b4150c6e1039b3372f0ab7e5f1


### List all Images on the Local Computer

- Notice that all the images have been removed.

In [28]:
!docker images

REPOSITORY                                           TAG                                                                           IMAGE ID       CREATED         SIZE
